In [29]:
import casadi as ca
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2
from matplotlib.patches import Ellipse
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import sys, os
sys.path.append(os.path.abspath('..'))
from se2 import se2_exp, se2_log, se2_Ad

In [30]:
def ES_EKF(X, P, v, w, dt, sigma_v, sigma_w, sigma_gps, z_gps, gps_flag):

    xi = np.array([v, 0, w])
    theta = np.arctan2(X[1, 0], X[0, 0])

    M = se2_exp(-np.array([v, 0, w]) * dt)   # 3x3 SE(2) matrix
    pose = np.array([M[0, 2],
                     M[1, 2],
                     np.arctan2(M[1, 0], M[0, 0])])
    A_k = se2_Ad(pose)

    H = np.array([
    [np.cos(theta), -np.sin(theta), 0],
    [np.sin(theta),  np.cos(theta), 0]
    ])
    Q_body = np.diag(np.array([sigma_v**2, 0, sigma_w**2]))
    Q_gps = np.diag(np.array([sigma_gps**2, sigma_gps**2]))

    # Propagate state
    X = X @ se2_exp(xi * dt)

    # Prediction
    P = A_k @ P @ A_k.T + Q_body*dt**2

    if not gps_flag:
        return X, P

    # Update
    p = np.array([X[0, 2], X[1, 2]])
    S = H @ P @ H.T + Q_gps
    K = P @ H.T @ np.linalg.inv(S)
    P = (np.eye(3) - K @ H) @ P @ (np.eye(3) - K @ H).T + K @ Q_gps @ K.T
    y = z_gps - p
    de = K @ y

    X = X @ se2_exp(de)

    return X, P

In [31]:
# Random generator
rng = np.random.default_rng(67676767)

# Constants
pi = np.pi

# Parameters
v_nom = 2
w_turn = 0.4
r_fig = 5
sigma_v = 1.0
sigma_w = 0.05
sigma_gps = 0.3
dt = 0.1

# Simulation parameters
T_lobe = 2 * pi / w_turn
T_fig8 = 10 * pi
T = 35
n_steps = int(T / dt)
gps_freq = 1

# Initial conditions
x0_true = np.array([0.0, 0.0, 0.0])
x0_est = np.array([0.0, 0.0, 0.0])
P0 = 0.1 * np.eye(3)
H = np.array([[1, 0, 0], [0, 1, 0]])
R = np.array([[sigma_gps**2, 0], [0, sigma_gps**2]])

In [32]:
x_true_hist = np.zeros((n_steps + 1, 3))
x_est_hist  = np.zeros((n_steps + 1, 3))
P_est_hist  = np.zeros((n_steps + 1, 3, 3))
z_hist      = np.full((n_steps, 2), np.nan)

x_true_hist[0] = x0_true
x_est_hist[0]  = x0_est
P_est_hist[0]  = P0

X_true = np.eye(3)
X_est = np.eye(3)
P_est = P0.copy()

for i in range(n_steps):
    t = i * dt
    if t % T_fig8 < T_lobe:
        w_ref = w_turn
    else:
        w_ref = -w_turn

    n_v = sigma_v * rng.standard_normal()
    n_w = sigma_w * rng.standard_normal()
    v_true = v_nom + n_v
    w_true = w_ref + n_w

    # True state (SE(2) Lie group propagation)
    xi = np.array([v_true, 0, w_true]) * dt
    X_true = X_true @ se2_exp(xi)
    x_true = np.array([X_true[0, 2], X_true[1, 2], np.arctan2(X_true[1, 0], X_true[0, 0])])

    if t % (1 / gps_freq) < 0.01:
        gps_flag = 1
        z_gps = H @ x_true + rng.multivariate_normal(np.zeros(2), R)
        z_hist[i] = z_gps
    else:
        z_gps = np.zeros(2)
        gps_flag = 0

    X_est, P_est = ES_EKF(X_est, P_est, v_nom, w_ref, dt, sigma_v, sigma_w, sigma_gps, z_gps, gps_flag)
    x_est = np.array([X_est[0, 2], X_est[1, 2], np.arctan2(X_est[1, 0], X_est[0, 0])])

    x_true_hist[i + 1] = x_true
    x_est_hist[i + 1]  = x_est
    P_est_hist[i + 1]  = P_est

In [33]:
def cov_ellipse(P_2x2, center, n_std=3.0):
    eigvals, eigvecs = np.linalg.eigh(P_2x2)
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    w, h = 2 * n_std * np.sqrt(eigvals)
    return Ellipse(xy=center, width=w, height=h, angle=angle,
                   fill=False, edgecolor='red', linewidth=1.5)

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_aspect('equal')
ax.grid(True)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('EKF Animation')

true_line, = ax.plot([], [], 'k-', linewidth=0.8, label='True')
est_line, = ax.plot([], [], 'b--', linewidth=1.2, label='EKF Estimate')
gps_scatter = ax.scatter([], [], c='green', s=15, marker='x', label='GPS', zorder=5, alpha=0.5)
ellipse_patch = None
ax.legend(loc='upper left')

pad = 2
ax.set_xlim(x_true_hist[:, 0].min() - pad, x_true_hist[:, 0].max() + pad)
ax.set_ylim(x_true_hist[:, 1].min() - pad, x_true_hist[:, 1].max() + pad)

skip = 5

def init():
    true_line.set_data([], [])
    est_line.set_data([], [])
    gps_scatter.set_offsets(np.empty((0, 2)))
    return true_line, est_line, gps_scatter

def animate(frame):
    global ellipse_patch
    k = frame * skip
    
    true_line.set_data(x_true_hist[:k+1, 0], x_true_hist[:k+1, 1])
    est_line.set_data(x_est_hist[:k+1, 0], x_est_hist[:k+1, 1])

    z_slice = z_hist[:k]
    gps_mask = ~np.isnan(z_slice[:, 0])
    if gps_mask.any():
        gps_scatter.set_offsets(z_slice[gps_mask])

    if ellipse_patch is not None:
        ellipse_patch.remove()
    P_k = P_est_hist[k]
    theta = x_est_hist[k, 2]
    R_th = np.array([[np.cos(theta), -np.sin(theta), 0],
                    [np.sin(theta),  np.cos(theta), 0],
                    [0, 0, 1]])
    P_world_xy = R_th @ P_k @ R_th.T
    ellipse_patch = cov_ellipse(P_world_xy[:2, :2], x_est_hist[k, :2])
    ax.add_patch(ellipse_patch)

    return true_line, est_line, gps_scatter, ellipse_patch

n_frames = (n_steps + 1) // skip
anim = FuncAnimation(fig, animate, init_func=init, frames=n_frames, interval=30, blit=False)
anim.save('es_ekf_animation.gif', writer='pillow', fps=30)
plt.close(fig)
print('Saved ekf_animation.gif')
HTML(anim.to_jshtml())

Saved ekf_animation.gif
